## Creación de Tensores

In [ ]:
"""
Preprocesamiento  del dataset de anestesia.

Este script lee todos los archivos CSV de cada conjunto (entrenamiento, validacion,
prueba), genera las ventanas deslizantes con relleno de ceros, normaliza
cada ventana con su propia media y desviacion estandar, y guarda el
resultado como tensores PyTorch (.pt) listos para cargarse directamente
en la GPU durante el entrenamiento.

Estructura de salida por conjunto:
    infusiones  : torch.float16  (N, 2, W)
    bis_hist    : torch.float16  (N, W)
    bis_hist_gt : torch.float32  (N, W)   -- sin normalizar, para la perdida
    covariables : torch.float32  (N, 4)
    etiquetas   : torch.float32  (N,)

Se usa float16 para las entradas normalizadas con el fin de reducir el
uso de memoria y acelerar la transferencia CPU → GPU.
"""

# =============================================================================
# CONFIGURACION
# =============================================================================
TAMANIYO_VENTANA   = 180

RUTA_ENTRENAMIENTO = r"C:\Users\yordi\Desktop\Entrenamiento_Limpios"
RUTA_VALIDACION    = r"C:\Users\yordi\Desktop\Validacion_Limpios"
RUTA_PRUEBA        = r"C:\Users\yordi\Desktop\Prueba_Limpios"

# Archivos de salida 
SALIDA_ENTRENAMIENTO = r"C:\Users\yordi\Desktop\dataset_entrenamiento.pt"
SALIDA_VALIDACION    = r"C:\Users\yordi\Desktop\dataset_validacion.pt"
SALIDA_PRUEBA        = r"C:\Users\yordi\Desktop\dataset_prueba.pt"


import os
import glob
import time
import numpy as np
import pandas as pd
import torch

# =============================================================================
# FUNCION DE NORMALIZACION POR VENTANA
# =============================================================================
def normalizar_ventana(ventana: np.ndarray) -> np.ndarray:
    """
    Normaliza un arreglo 1D con su media y desviacion estandar.
    Si la desviacion estandar es practicamente nula, solo se centra.
    """
    media      = ventana.mean()
    desviacion = ventana.std()
    if desviacion < 1e-8:
        return ventana - media
    return (ventana - media) / desviacion


# =============================================================================
# FUNCION PRINCIPAL DE CONVERSION
# =============================================================================
def procesar_carpeta(ruta_carpeta: str, nombre_conjunto: str) -> dict:
    """
    Lee todos los archivos CSV, genera las ventanas
    deslizantes y devuelve un diccionario de tensores empaquetados.

    Parametros
    ----------
    ruta_carpeta   : Ruta al directorio con archivos CSV.
    nombre_conjunto: Nombre para mostrar en la consola 

    Retorna
    -------
    Diccionario con las claves:
        'infusiones'  : (N, 2, W)  float16
        'bis_hist'    : (N, W)     float16
        'bis_hist_gt' : (N, W)     float32
        'covariables' : (N, 4)     float32
        'etiquetas'   : (N,)       float32
    """
    archivos = sorted(glob.glob(os.path.join(ruta_carpeta, "*.csv")))
    if not archivos:
        raise FileNotFoundError(
            f"No se encontraron archivos CSV en: {ruta_carpeta}"
        )

    print(f"\n[{nombre_conjunto.upper()}] {len(archivos)} archivos encontrados.")

    # Listas que acumulan los datos de todos los pacientes
    lista_infusiones  = []
    lista_bis_hist    = []
    lista_bis_hist_gt = []
    lista_covariables = []
    lista_etiquetas   = []

    W = TAMANIYO_VENTANA

    for idx_archivo, ruta in enumerate(archivos):
        df = pd.read_csv(ruta, usecols=[
            "PPF20_RATE", "RFTN20_RATE", "BIS",
            "Edad_Norm", "Peso_norm", "Altura_norm", "Sexo_norm"
        ])

        propofol    = df["PPF20_RATE"].to_numpy(dtype=np.float32)
        remifent    = df["RFTN20_RATE"].to_numpy(dtype=np.float32)
        bis         = df["BIS"].to_numpy(dtype=np.float32)
        covariables = np.array([
            df["Edad_Norm"].iloc[0],
            df["Peso_norm"].iloc[0],
            df["Altura_norm"].iloc[0],
            df["Sexo_norm"].iloc[0],
        ], dtype=np.float32)

        T = len(bis)

        # Pre-alocar arreglos para este paciente
        inf_paciente  = np.empty((T, 2, W), dtype=np.float16)
        bh_paciente   = np.empty((T, W),    dtype=np.float16)
        bh_gt_paciente = np.empty((T, W),   dtype=np.float32)
        eta_paciente  = np.empty(T,          dtype=np.float32)

        for t in range(T):
            inicio = t - W + 1
            fin    = t + 1          # Exclusivo

            if inicio < 0:
                n_relleno  = -inicio
                v_prop     = np.concatenate(
                    [np.zeros(n_relleno, dtype=np.float32), propofol[:fin]]
                )
                v_remi     = np.concatenate(
                    [np.zeros(n_relleno, dtype=np.float32), remifent[:fin]]
                )
                v_bis      = np.concatenate(
                    [np.zeros(n_relleno, dtype=np.float32), bis[:fin]]
                )
            else:
                v_prop = propofol[inicio:fin].copy()
                v_remi = remifent[inicio:fin].copy()
                v_bis  = bis[inicio:fin].copy()

            # Guardar BIS sin normalizar (para la perdida auxiliar MSE)
            bh_gt_paciente[t] = v_bis

            # Normalizar cada ventana con su propia estadistica
            v_prop_n = normalizar_ventana(v_prop).astype(np.float16)
            v_remi_n = normalizar_ventana(v_remi).astype(np.float16)
            v_bis_n  = normalizar_ventana(v_bis).astype(np.float16)

            inf_paciente[t, 0, :] = v_prop_n
            inf_paciente[t, 1, :] = v_remi_n
            bh_paciente[t]        = v_bis_n
            eta_paciente[t]       = bis[t]

        lista_infusiones.append(inf_paciente)
        lista_bis_hist.append(bh_paciente)
        lista_bis_hist_gt.append(bh_gt_paciente)
        lista_covariables.append(
            np.tile(covariables, (T, 1))    # (T, 4)
        )
        lista_etiquetas.append(eta_paciente)

        # Reporte de progreso por archivo
        print(
            f"  Archivo [{idx_archivo + 1:3d}/{len(archivos)}]: "
            f"{os.path.basename(ruta):40s} | "
            f"{T:5d} pasos → {T} ventanas acumuladas"
        )

    # Concatenacion de todos los pacientes en un unico tensor
    print(f"\n  Concatenando arreglos del conjunto {nombre_conjunto}...")
    t0 = time.time()

    inf_total  = np.concatenate(lista_infusiones,  axis=0)   # (N, 2, W)
    bh_total   = np.concatenate(lista_bis_hist,    axis=0)   # (N, W)
    bh_gt_tot  = np.concatenate(lista_bis_hist_gt, axis=0)   # (N, W)
    cov_total  = np.concatenate(lista_covariables, axis=0)   # (N, 4)
    eta_total  = np.concatenate(lista_etiquetas,   axis=0)   # (N,)

    print(
        f"  Concatenacion completada en {time.time() - t0:.1f} s. "
        f"Total de muestras: {len(eta_total):,}"
    )

    # Conversion a tensores de PyTorch
    dataset = {
        "infusiones":  torch.from_numpy(inf_total),
        "bis_hist":    torch.from_numpy(bh_total),
        "bis_hist_gt": torch.from_numpy(bh_gt_tot),
        "covariables": torch.from_numpy(cov_total),
        "etiquetas":   torch.from_numpy(eta_total),
    }

    # Reporte de memoria 
    mb_total = sum(v.nbytes for v in dataset.values()) / (1024 ** 2)
    print(f"  Memoria aproximada del dataset: {mb_total:.1f} MB")

    return dataset


# =============================================================================
# PUNTO DE ENTRADA
# =============================================================================
if __name__ == "__main__":
    tiempo_inicio_global = time.time()

    conjuntos = [
        (RUTA_ENTRENAMIENTO, SALIDA_ENTRENAMIENTO, "entrenamiento"),
        (RUTA_VALIDACION,    SALIDA_VALIDACION,    "validacion"),
        (RUTA_PRUEBA,        SALIDA_PRUEBA,        "prueba"),
    ]

    for ruta_entrada, ruta_salida, nombre in conjuntos:
        t0 = time.time()
        dataset = procesar_carpeta(ruta_entrada, nombre)

        print(f"\n  Guardando en: {ruta_salida}")
        torch.save(dataset, ruta_salida)
        print(
            f"  Guardado completado en {time.time() - t0:.1f} s."
        )

    print(
        f"\n[INFO] Preprocesamiento total finalizado en "
        f"{time.time() - tiempo_inicio_global:.1f} s."
    )
    print("[INFO] Los archivos .pt estan listos para el entrenamiento.")


[ENTRENAMIENTO] 480 archivos encontrados.
  Archivo [  1/480]: Paciente_1001_limpio_ConFiltro.csv       |   856 pasos → 856 ventanas acumuladas
  Archivo [  2/480]: Paciente_1036_limpio_ConFiltro.csv       |   651 pasos → 651 ventanas acumuladas
  Archivo [  3/480]: Paciente_1043_limpio_ConFiltro.csv       |   644 pasos → 644 ventanas acumuladas
  Archivo [  4/480]: Paciente_1047_limpio_ConFiltro.csv       |  1343 pasos → 1343 ventanas acumuladas
  Archivo [  5/480]: Paciente_104_limpio_ConFiltro.csv        |  1262 pasos → 1262 ventanas acumuladas
  Archivo [  6/480]: Paciente_1073_limpio_ConFiltro.csv       |  1327 pasos → 1327 ventanas acumuladas
  Archivo [  7/480]: Paciente_1089_limpio_ConFiltro.csv       |  1209 pasos → 1209 ventanas acumuladas
  Archivo [  8/480]: Paciente_1098_limpio_ConFiltro.csv       |   799 pasos → 799 ventanas acumuladas
  Archivo [  9/480]: Paciente_1126_limpio_ConFiltro.csv       |   583 pasos → 583 ventanas acumuladas
  Archivo [ 10/480]: Paciente_1146_